In [2]:
from utils import *

n_sim = 1000
n = int(1000.5/0.7)
B_RF = int(2000 )

seed = 42
n_covariates = 3

X_pred_point = pd.DataFrame({'X_1': [40], 'X_2': [1], 'X_3': [80]})
data_generation_parameter =    { 'shape_weibull': 1,  'p_1': -0.4, 'p_2': -0.2, 'p_3': -0.1, 'n': n,
                                    'scale_weibull_base':   5_329_174_608_059        , 
                                    'rate_censoring':       0.04477    , 
                                    'tau': 20,
                                    'X_pred_point': X_pred_point}  
tau= data_generation_parameter['tau']


params_rf =         {   'n_estimators':B_RF,                        
                        'max_depth':4,
                        'min_samples_split':5,
                        'max_features': 'log2',
                        'random_state':  seed,
                        'weighted_bootstrapping': True, }

data = create_weibull_data(params=data_generation_parameter, random_state=seed)
df_train, df_test = stratified_split(data=data, random_state=seed, test_size=0.3)
df_train = create_data_with_ipc_weights(data=df_train, params=data_generation_parameter)
df_test = create_data_with_ipc_weights(data=df_test, params=data_generation_parameter)

### Random Forest Modell ###
# Fit
params_rf["random_state"] = seed
clf = DecisionTreeBaggingClassifier(params_rf)
clf.fit(
    X=df_train.drop(
        ["time", "event", "weights_ipcw", "survived"], axis=1
    ).values,
    y=df_train["survived"].values,
    sample_weights=df_train["weights_ipcw"].values,
)

# Evaluation auf Testdaten
_ , pred  =clf.predict_proba(df_test.drop(
    ["weights_ipcw", "survived", "time", "event"], axis=1
).values)
rf_test_mse = ipc_weighted_mse(
    y_true=df_test["survived"].values,
    y_pred=pred,
    sample_weight=df_test["weights_ipcw"].values,
)

# Prediction für X_erwartung
_ ,rf_pred = clf.predict_proba(data_generation_parameter['X_pred_point'].values)


In [ ]:
B_1 = 2

np_train = df_train.values
df_train_columns_name = df_train.columns
preds = np.empty(B_1)

rng = np.random.default_rng(seed)
first_level_boot_indices = rng.choice(
    a=np.arange(df_train.shape[0]), size=(B_1, df_train.shape[0]), replace=True
)

for b in range(B_1):

    np_train_boot = np_train[first_level_boot_indices[b], :]

    # Create the new dataset with IPCW weights
    df_train_boot = create_data_with_ipc_weights(
        data=pd.DataFrame(np_train_boot, columns=df_train_columns_name), params=data_generation_parameter
    )

    # Set the random state and fit the classifier
    clf = DecisionTreeBaggingClassifier(params_rf)
    clf.set_random_state(random_state=seed + 1000+ b )
    
    clf.fit(
        X=df_train_boot.drop(
            ["time", "event", "weights_ipcw", "survived"], axis=1
        ).values,
        y=df_train_boot["survived"].values,
        sample_weights=df_train_boot["weights_ipcw"].values,
    )
    
    # Predict and store result
    _ ,pred = clf.predict_proba(X_pred_point.values)
    preds[b] = pred[0]

# Calculate variance using numpy
np.var(preds, ddof=1)

,X_1,X_2,X_3,event,time,survived,weights_ipcw
0,35.602452,1,80.921771,False,0.596156,999,0.0
1,39.912186,1,94.408126,False,1.398367,999,0.0
2,30.690774,1,99.154347,False,17.715771,999,0.0
3,44.139941,1,78.860836,True,12.591088,0,0.001734
4,44.227416,1,77.861444,False,1.14927,999,0.0
...,...,...,...,...,...,...,...
995,42.24162,1,79.349543,False,2.321686,999,0.0
996,42.357693,1,73.435139,False,14.775805,999,0.0
997,25.177356,1,84.879914,False,48.39443,1,0.002494
998,38.994477,1,93.069996,False,22.480295,1,0.002494


In [ ]:



    # Create the new dataset with IPCW weights
    df_train_boot = create_new_dataset_with_ipcw_weights(
        data=pd.DataFrame(np_train_boot, columns=df_train_columns_name), t=tau
    )

    # Set the random state and fit the classifier
    clf = DecisionTreeBaggingClassifier(params_rf)
    clf.set_random_state(random_state=seed + 1000+ b )
    
    clf.fit(
        X=df_train_boot.drop(
            ["time", "event", "weights_ipcw", "survived"], axis=1
        ).values,
        y=df_train_boot["survived"].values,
        sample_weights=df_train_boot["weights_ipcw"].values,
    )
    
    # Predict and store result
    _ ,pred = clf.predict_proba(X_pred_point.values)
    preds[b] = pred[0]

# Calculate variance using numpy
np.var(preds, ddof=1)